In [ ]:
# Setup and helper functions moved to external module
from equations_file import *

# If you change the external module and want live reload during development, uncomment:
# %load_ext autoreload
# %autoreload 2

In [ ]:
save_dir = "/Users/simoncorrera/Desktop/MQ NGC3201 project copy/sequoia paper matieral/thesis only figures"

In [ ]:
with fitsio.FITS(galah_Gaia_fits) as hdul1:
    data_1 = hdul1[1].read()
galah_Gaia_raw = pd.DataFrame(data_1)
galah_Gaia_raw = ensure_native_endian(galah_Gaia_raw)

In [ ]:
galah_Gaia = galah_Gaia_raw[(galah_Gaia_raw['snr_px_ccd3'] > 30)       # Kushniruk 2026 also has cuts in log g and temp. I wonder if I should as well
                        & (galah_Gaia_raw['flag_sp'] == 0)
                        & (galah_Gaia_raw['flag_fe_h'] == 0)
                        & (galah_Gaia_raw['e_fe_h'] < 0.1)
                        & (galah_Gaia_raw['flag_sp_fit'] == 0)
                        & (galah_Gaia_raw['flag_red'] == 0)
                        & (galah_Gaia_raw['ruwe'] < 1.4)
                        & (galah_Gaia_raw['logg'] < 3.5)
                        & (galah_Gaia_raw['teff'] > 4000)
                        & (galah_Gaia_raw['teff'] < 6500)
                        # & (galah_Gaia_raw['flag_mg_fe'] == 0)
                        # & (galah_Gaia_raw['flag_na_fe'] == 0)
                        # & (galah_Gaia_raw['flag_cu_fe'] == 0)
                        & (galah_Gaia_raw['fe_h'] < -0.8)
                        # & (galah_Gaia_raw['fe_h'] < -1.24)  # the 16th and 84th percentile spreads of all flagged elements sequoia Fe/H: -1.24, -1.72
                        # & (galah_Gaia_raw['fe_h'] > -1.72) # the 16th and 84th percentile spreads of only Fe/h flagged elements sequoia Fe/H: -1.28, -2.01
                        # & (galah_Gaia_raw['ti_fe'] > 0.25)
                        
                        & (galah_Gaia_raw['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main',
                                                               'k2_hermes', 'galah_phase2', 'tess_hermes'
                                                               ]))
                        & (galah_Gaia_raw['radial_velocity'].notnull())
                        & (galah_Gaia_raw['ra_gaia'].notnull())
                        & (galah_Gaia_raw['dec_gaia'].notnull())
                        & (galah_Gaia_raw['pmra_gaia'].notnull())
                        & (galah_Gaia_raw['pmdec_gaia'].notnull())
                        & (galah_Gaia_raw['parallax_gaia'] > 0)
                        
    ]

print(f"Number of stars after cuts: {len(galah_Gaia)}")

In [ ]:
NGC3201_galah = pd.merge( galah_Gaia, Vmans_NGC3201_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC5139_galah = pd.merge( galah_Gaia, Vmans_NGC5139_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC1851_galah = pd.merge( galah_Gaia, Vmans_NGC1851_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')   
NGC0104_galah = pd.merge( galah_Gaia, Vmans_NGC0104_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')

galah_Kushniruk_GSE = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])-35)**2) / (17.5**2)) + (((galah_Gaia['jphi'] + 130)**2) / (250**2)) < 1)
                                #  & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                 & (galah_Gaia_raw['flag_mg_fe'] == 0)
                                 & (galah_Gaia_raw['flag_na_fe'] == 0)
                                 & (galah_Gaia_raw['flag_cu_fe'] == 0)
                                 & ((galah_Gaia_raw['mg_fe'] - galah_Gaia_raw['cu_fe']) - ((1.06923) * galah_Gaia_raw['na_fe']) > 0.40150)
                                 ]

# Kushniruk et al . 2026 https://doi.org/10.1051/0004-6361/202451201
galah_Kushniruk_Thamnos1 = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])- 13.5)**2) / (4.0**2)) + (((galah_Gaia['jphi'] + 600)**2) / (350**2)) < 1)
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                        & (galah_Gaia_raw['fe_h'] < -0.8)
                                      ]

galah_Kushniruk_Thamnos2 = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])-8.6)**2) / (3.5**2)) + (((galah_Gaia['jphi'] + 1184)**2) / (175**2)) < 1)
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                        & (galah_Gaia_raw['fe_h'] < -0.8)
                                      ]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_GSE = galah_Gaia[GSE_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            & (galah_Gaia_raw['fe_h'] < -0.8)
                            ]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_Sequoia = galah_Gaia[Sequoia_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            & (galah_Gaia_raw['fe_h'] < -0.8)
                            # & (galah_Gaia_raw['fe_h'] > -1.3)

                            # & (galah_Gaia_raw['ca_fe'] < 0.37)
]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_Sequoia_p_thamnos = galah_Gaia[Sequoia_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            & (galah_Gaia_raw['fe_h'] < -0.8)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id']))

                            # & (galah_Gaia_raw['y_fe'] < 0.8)
]

galah_Gaia_Halo = galah_Gaia[(np.sqrt((galah_Gaia['vphi']-230)**2 + (galah_Gaia['vr'])**2 + (galah_Gaia['vz'])**2) > 230) # Koppelman et al. 2019  https://doi.org/10.1051/0004-6361/201936738
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id'])) 
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id'])) 
                            # & (galah_Gaia_raw['fe_h'] < -1.1)
                            # & (galah_Gaia_raw['fe_h'] > -1.9)
                            & (galah_Gaia_raw['fe_h'] < -0.8)
                             ]


In [ ]:
Halo_Pradosh_galah_Gaia = galah_Gaia[
                                            (galah_Gaia['fe_h'] < -0.8)&
                                            (galah_Gaia['ti_fe'] > 0.25)
                                            & (galah_Gaia['flag_ti_fe'] == 0)
                                            & (galah_Gaia['Lz'] > -1500)
                                            & (galah_Gaia['Lz'] < 1500)
                                            & (galah_Gaia['ecc_gaia'] > 0.5)
                                            & (galah_Gaia['ecc_gaia'] < 0.8)
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC3201_new_ratios['gaiadr3_source_id']))
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC5139_new_ratios['gaiadr3_source_id']))
                                            ]

thickdisk_galah_Gaia = galah_Gaia[(galah_Gaia['fe_h'] < -0.8)
                                                 & (galah_Gaia['ti_fe'] > 0.25)
                                                 & (galah_Gaia['Lz'] > 0)
                                                 & (galah_Gaia['ecc_gaia'] < 0.5)
                                                #  & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                                                # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC3201_new_ratios['gaiadr3_source_id']))
                                                # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC5139_new_ratios['gaiadr3_source_id']))
                                            ]
print(len(Halo_Pradosh_galah_Gaia))
print(len(thickdisk_galah_Gaia))

galah_Gaia_Halo = Halo_Pradosh_galah_Gaia

In [ ]:
# print(f'len(NGC3201_galah): {len(NGC3201_galah)}')
# print(f'len(NGC5139_galah): {len(NGC5139_galah)}')
# print(f'len(NGC1851_galah): {len(NGC1851_galah)}')
# print(f'len(NGC0104_galah): {len(NGC0104_galah)}')
# print(f'len(galah_Kushniruk_GSE): {len(galah_Kushniruk_GSE)}')
print(f'len(galah_Kushniruk_Thamnos1): {len(galah_Kushniruk_Thamnos1)}')
print(f'len(galah_Kushniruk_Thamnos2): {len(galah_Kushniruk_Thamnos2)}')
print(f'len(galah_Gaia_GSE): {len(galah_Gaia_GSE)}')
print(f'len(galah_Gaia_Sequoia): {len(galah_Gaia_Sequoia)}')
print(f'len(galah_Gaia_Sequoia_p_thamnos): {len(galah_Gaia_Sequoia_p_thamnos)}')
print(f'len(galah_Gaia_Halo): {len(galah_Gaia_Halo)}')
# print(f'len(galah_Gaia_else): {len(galah_Gaia_else)}')
print(f'galah_Gaia_new_ratios: {len(galah_Gaia)}')

In [ ]:

element_list_partial_small_galah_7 = [ 'fe', 'cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_2 = ['cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_3 = ['fe', 'v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_7_4 = ['fe', 'si', 'nd', 'na', 'mg', 'k', 'ca', 'sc', 'ti', 'v', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba']
element_list_partial_small_galah_7_5 = [ 'v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']  # now I wonder if I could use sm_fe, o_fe, 
element_list_partial_small_galah_10 = ['fe', 'v', 'sc', 'si', 'ca', 'mn', 'na', 'nd']                                                             # and/or la_fe
element_list_partial_small_galah_11 = ['y', 'ba', 'nd', 'eu']                                                                                 # also la and sm would bring me to (50, 45, or 44combined)/65 sequoia stars otherwise
element_list_partial_small_galah_16 =  ["na", "mg", "ca", "ti", "cr" ,"mn", "ni", 'cu', 'y', "ba", "nd"] #This is what I used in my Poster
element_list_partial_small_galah_17 =  ['fe', "na", "mg", "ca", "ti", "cr" ,"mn", "ni", 'cu', 'y', "ba", "nd"]
element_list_partial_small_galah_18 =  ['fe', "na", "mg", 'si', "ca", "ti", "mn", "ni", 'cu', 'y', "ba", "nd"]


ASA_Plots_ratios = ['fe', 'si', 'ca', 'ti', 'mn', 'ni', 'y', 'nd']
S_VS_G_best_elms = ['na', 'mn', 'ti', 'cr']

# element_list = S_VS_G_best_elms
element_list = element_list_partial_small_galah_18


galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia, element_list)

galah_Gaia_new_ratios = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0] # this ontains a full catalog from galah but now it includes ratios for Ca, Si, Ti, Ni, Zr, Ce, and Nd relative to eachother: Pradosh did recomend taking out
                                                                 # Si, and Ni whitch I can do by changing "element_list_partial_large" to element_list_partial_small
galah_gaia_new_element_ratio_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[1]     # this just lists the new columns ratios so I can more easily call them
galah_gaia_new_element_ratio_errors_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[2] 

galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_GSE, element_list)
galah_Gaia_GSE_new_ratios = galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_GSE, element_list)
galah_Gaia_Kushniruk_GSE_new_ratios = galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Sequoia, element_list)
galah_Gaia_Sequoia_new_ratios = galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Sequoia_p_Thamnos_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Sequoia_p_thamnos, element_list)
galah_Gaia_Sequoia_p_Thamnos_new_ratios = galah_Gaia_Sequoia_p_Thamnos_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC3201_galah, element_list)
# galah_Gaia_NGC3201_new_ratios = galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC5139_galah, element_list)
galah_Gaia_NGC5139_new_ratios = galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos1, element_list)
galah_Gaia_Thamnos1_new_ratios = galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos2, element_list)
galah_Gaia_Thamnos2_new_ratios = galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC1851_galah, element_list)
# galah_Gaia_NGC1851_new_ratios = galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Halo, element_list)
galah_Gaia_Halo_new_ratios = galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist[0]

Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(Halo_Pradosh_galah_Gaia, element_list)
Halo_Pradosh_galah_Gaia_new_ratios = Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_else_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_else, element_list)
# galah_Gaia_else_new_ratios = galah_Gaia_else_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_raw_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_raw, element_list)
galah_Gaia_raw_new_ratios = galah_Gaia_raw_new_ratios_catolog_ratiolist_ratioerrorlist[0]


'----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------'

# element_list_full_galah_almost = ["n", "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti", "v", "cr" ,"mn", "co", "ni", "cu", "zn", "y", "zr", "ba", "la", "ce", "nd", "sm", "eu"]# there is nn_li in galah (nural network Li/Fe) but I dont want to include that right now.
element_list_full_galah_almost = ['fe', 'c', 'n', "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti", "v", "cr" ,"mn", "co", "ni", "cu", "zn", 
                                  'rb', 'sr', 
                                  "y", "zr", 
                                  'mo', 'ru',
                                  "ba", "la", "ce", "nd", "sm", "eu"]# there is nn_li in galah (nural network Li/Fe) but I dont want to include that right now.


element_list_nessessary_flags = element_list_full_galah_almost

galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia, element_list_nessessary_flags)
galah_Gaia_new_ratios_nessessary_flags_ratio_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[1]
galah_Gaia_new_ratios_nessessary_flags = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0]


galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_Sequoia, element_list_nessessary_flags)
galah_Gaia_Sequoia_new_ratios_nessessary_flags = galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist[0]
 
galah_Gaia_Sequoia_p_Thamnos_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_Sequoia_p_thamnos, element_list_nessessary_flags)
galah_Gaia_Sequoia_p_Thamnos_new_ratios_nessessary_flags = galah_Gaia_Sequoia_p_Thamnos_new_ratios_catolog_ratiolist_ratioerrorlist[0]


galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_GSE, element_list_nessessary_flags)
galah_Gaia_GSE_new_ratios_nessessary_flags = galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos_1_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Kushniruk_Thamnos1, element_list_nessessary_flags)
galah_Gaia_Thamnos_1_new_ratios_nessessary_flags = galah_Gaia_Thamnos_1_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos_2_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Kushniruk_Thamnos2, element_list_nessessary_flags)
galah_Gaia_Thamnos_2_new_ratios_nessessary_flags = galah_Gaia_Thamnos_2_new_ratios_catolog_ratiolist_ratioerrorlist[0]


galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_Halo, element_list_nessessary_flags)
galah_Gaia_Halo_new_ratios_nessessary_flags = galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist[0]

Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(Halo_Pradosh_galah_Gaia, element_list_nessessary_flags)
Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags = Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(NGC5139_galah, element_list_nessessary_flags)
galah_Gaia_NGC5139_new_ratios_nessessary_flags = galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist[0]



In [ ]:
print(f'len(galah_Gaia_Sequoia_new_ratios): {len(galah_Gaia_Sequoia_new_ratios)}')
print(f'len(galah_Gaia_Sequoia_p_Thamnos_new_ratios): {len(galah_Gaia_Sequoia_p_Thamnos_new_ratios)}')
print(f'len(galah_Gaia_GSE_new_ratios): {len(galah_Gaia_GSE_new_ratios)}')
print(f'len(galah_Gaia_Halo_new_ratios): {len(galah_Gaia_Halo_new_ratios)}')
print(f'len(galah_Gaia_else_new_ratios): {len(galah_Gaia_new_ratios)}')
# print(f'len(galah_Gaia_else_new_ratios): {len(galah_Gaia_else_new_ratios)}')


In [ ]:
Xs = ['c', 'n', "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti", "v", "cr" ,"mn", "co", "ni", "cu", "zn", 
                                  'rb', 'sr', 
                                  "y", "zr", 
                                  'mo', 'ru',
                                  "ba", "la", "ce", "nd", "sm", "eu"]
for X in Xs:
    X_fe = X + '_fe'
    X_H = X + '_h'

    galah_Gaia_new_ratios_nessessary_flags[X_H] = galah_Gaia_new_ratios_nessessary_flags[X_fe] + galah_Gaia_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_Sequoia_new_ratios_nessessary_flags[X_H] = galah_Gaia_Sequoia_new_ratios_nessessary_flags[X_fe] + galah_Gaia_Sequoia_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_Sequoia_p_Thamnos_new_ratios_nessessary_flags[X_H] = galah_Gaia_Sequoia_p_Thamnos_new_ratios_nessessary_flags[X_fe] + galah_Gaia_Sequoia_p_Thamnos_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_GSE_new_ratios_nessessary_flags[X_H] = galah_Gaia_GSE_new_ratios_nessessary_flags[X_fe] + galah_Gaia_GSE_new_ratios_nessessary_flags['fe_h']
    Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags[X_H] = Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags[X_fe] + Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_Thamnos_1_new_ratios_nessessary_flags[X_H] = galah_Gaia_Thamnos_1_new_ratios_nessessary_flags[X_fe] + galah_Gaia_Thamnos_1_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_Thamnos_2_new_ratios_nessessary_flags[X_H] = galah_Gaia_Thamnos_2_new_ratios_nessessary_flags[X_fe] + galah_Gaia_Thamnos_2_new_ratios_nessessary_flags['fe_h']
   
    
    galah_Gaia_new_ratios[X_H] = galah_Gaia_new_ratios[X_fe] + galah_Gaia_new_ratios['fe_h']
    galah_Gaia_Sequoia_new_ratios[X_H] = galah_Gaia_Sequoia_new_ratios[X_fe] + galah_Gaia_Sequoia_new_ratios['fe_h']
    galah_Gaia_Sequoia_p_Thamnos_new_ratios[X_H] = galah_Gaia_Sequoia_p_Thamnos_new_ratios[X_fe] + galah_Gaia_Sequoia_p_Thamnos_new_ratios['fe_h']
    galah_Gaia_GSE_new_ratios[X_H] = galah_Gaia_GSE_new_ratios[X_fe] + galah_Gaia_GSE_new_ratios['fe_h']
    Halo_Pradosh_galah_Gaia_new_ratios[X_H] = Halo_Pradosh_galah_Gaia_new_ratios[X_fe] + Halo_Pradosh_galah_Gaia_new_ratios['fe_h']
    galah_Gaia_Thamnos1_new_ratios[X_H] = galah_Gaia_Thamnos1_new_ratios[X_fe] + galah_Gaia_Thamnos1_new_ratios['fe_h']
    galah_Gaia_Thamnos2_new_ratios[X_H] = galah_Gaia_Thamnos2_new_ratios[X_fe] + galah_Gaia_Thamnos2_new_ratios['fe_h']
    
for X in element_list:
    
    cut_off = 0.10
    if X == 'fe':
        continue
    if X in ['y', 'ba', 'la', 'ce', 'nd', 'sm', 'eu']:
        cut_off = 0.12
    # print(X, cut_off)
    X_fe = X + '_fe'
    e_X_fe = 'e_' + X_fe
    galah_Gaia_new_ratios = galah_Gaia_new_ratios[galah_Gaia_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Sequoia_new_ratios = galah_Gaia_Sequoia_new_ratios[galah_Gaia_Sequoia_new_ratios[e_X_fe] < cut_off]
    Halo_Pradosh_galah_Gaia_new_ratios = Halo_Pradosh_galah_Gaia_new_ratios[Halo_Pradosh_galah_Gaia_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_GSE_new_ratios = galah_Gaia_GSE_new_ratios[galah_Gaia_GSE_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Halo_new_ratios = galah_Gaia_Halo_new_ratios[galah_Gaia_Halo_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Thamnos1_new_ratios = galah_Gaia_Thamnos1_new_ratios[galah_Gaia_Thamnos1_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Thamnos2_new_ratios = galah_Gaia_Thamnos2_new_ratios[galah_Gaia_Thamnos2_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Sequoia_p_Thamnos_new_ratios = galah_Gaia_Sequoia_p_Thamnos_new_ratios[galah_Gaia_Sequoia_p_Thamnos_new_ratios[e_X_fe] < cut_off]
    print(f'{X}, {cut_off}: {len(galah_Gaia_Sequoia_new_ratios)}')
print(f'Seq_lim: {len(galah_Gaia_Sequoia_new_ratios)}')
print(f'Seq_Tham: {len(galah_Gaia_Sequoia_p_Thamnos_new_ratios)}')
print(f'P_Halo: {len(Halo_Pradosh_galah_Gaia_new_ratios)}')
print(f'GSE: {len(galah_Gaia_GSE_new_ratios)}')
print(f'Halo: {len(galah_Gaia_Halo_new_ratios)}')
print(f'Thamnos1: {len(galah_Gaia_Thamnos1_new_ratios)}')
print(f'Thamnos2: {len(galah_Gaia_Thamnos2_new_ratios)}')
# print(f'Else: {len(galah_Gaia_else_new_ratios)}')
    

In [ ]:
''' THESE INCLUDE elm list 18 and 3d '''
'''  SL v G, SL v PH, ST v G, ST v PH, G v PH
'''

RFC_Seq_lim_vs_GSE_location = '/Users/simoncorrera/Desktop/MQ NGC3201 project copy/sequoia paper matieral/RFC_results/Seq_lim_GSE_elmlist_18_RFC_test.csv'
RFC_Seq_lim_vs_GSE_raw = pd.read_csv(RFC_Seq_lim_vs_GSE_location)

RFC_Seq_Tham_vs_GSE_location = '/Users/simoncorrera/Desktop/MQ NGC3201 project copy/sequoia paper matieral/RFC_results/Seq_Tham_vs_GSE_elmlist_18_RFC_test.csv'
RFC_Seq_Tham_vs_GSE_raw = pd.read_csv(RFC_Seq_Tham_vs_GSE_location)

RFC_Seq_lim_vs_PHalo_location = '/Users/simoncorrera/Desktop/MQ NGC3201 project copy/sequoia paper matieral/RFC_results/Seq_lim_vs_Halo_elmlist_18_RFC_test.csv'
RFC_Seq_lim_vs_PHalo_raw = pd.read_csv(RFC_Seq_lim_vs_PHalo_location)

RFC_Seq_Tham_vs_PHalo_location = '/Users/simoncorrera/Desktop/MQ NGC3201 project copy/sequoia paper matieral/RFC_results/Seq_Tham_vs_Halo_elmlist_18_RFC_test.csv'
RFC_Seq_Tham_vs_PHalo_raw = pd.read_csv(RFC_Seq_Tham_vs_PHalo_location)

RFC_GSE_vs_PHalo_location = '/Users/simoncorrera/Desktop/MQ NGC3201 project copy/sequoia paper matieral/RFC_results/GSE_vs_Halo_elmlist_18_RFC_test.csv'
RFC_GSE_vs_PHalo_raw = pd.read_csv(RFC_GSE_vs_PHalo_location)


In [ ]:
X1_X2 = 'fe_h'
Y1_Y2 = 'ti_fe'

X1_X2_mask_GH= ((RFC_GSE_vs_PHalo_raw['ratio1'] != X1_X2) & (RFC_GSE_vs_PHalo_raw['ratio2'] != X1_X2) & (RFC_GSE_vs_PHalo_raw['ratio3'] != X1_X2))
X1_X2_mask_SLH = ((RFC_Seq_lim_vs_PHalo_raw['ratio1'] != X1_X2) & (RFC_Seq_lim_vs_PHalo_raw['ratio2'] != X1_X2) & (RFC_Seq_lim_vs_PHalo_raw['ratio3'] != X1_X2))  
X1_X2_mask_STJ = ((RFC_Seq_Tham_vs_PHalo_raw['ratio1'] != X1_X2) & (RFC_Seq_Tham_vs_PHalo_raw['ratio2'] != X1_X2) & (RFC_Seq_Tham_vs_PHalo_raw['ratio3'] != X1_X2))
X1_X2_mask_SLG = ((RFC_Seq_lim_vs_GSE_raw['ratio1'] != X1_X2) & (RFC_Seq_lim_vs_GSE_raw['ratio2'] != X1_X2) & (RFC_Seq_lim_vs_GSE_raw['ratio3'] != X1_X2))
X1_X2_mask_STG = ((RFC_Seq_Tham_vs_GSE_raw['ratio1'] != X1_X2) & (RFC_Seq_Tham_vs_GSE_raw['ratio2'] != X1_X2) & (RFC_Seq_Tham_vs_GSE_raw['ratio3'] != X1_X2))  
  
Y1_Y2_mask_GH= ((RFC_GSE_vs_PHalo_raw['ratio1'] != Y1_Y2) & (RFC_GSE_vs_PHalo_raw['ratio2'] != Y1_Y2) & (RFC_GSE_vs_PHalo_raw['ratio3'] != Y1_Y2))
Y1_Y2_mask_SLH = ((RFC_Seq_lim_vs_PHalo_raw['ratio1'] != Y1_Y2) & (RFC_Seq_lim_vs_PHalo_raw['ratio2'] != Y1_Y2) & (RFC_Seq_lim_vs_PHalo_raw['ratio3'] != Y1_Y2))  
Y1_Y2_mask_STJ = ((RFC_Seq_Tham_vs_PHalo_raw['ratio1'] != Y1_Y2) & (RFC_Seq_Tham_vs_PHalo_raw['ratio2'] != Y1_Y2) & (RFC_Seq_Tham_vs_PHalo_raw['ratio3'] != Y1_Y2))  

RFC_GSE_vs_PHalo_raw_copy = RFC_GSE_vs_PHalo_raw[X1_X2_mask_GH][Y1_Y2_mask_GH]
RFC_Seq_lim_vs_PHalo_raw_copy = RFC_Seq_lim_vs_PHalo_raw[X1_X2_mask_SLH][Y1_Y2_mask_SLH]
RFC_Seq_Tham_vs_PHalo_raw_copy = RFC_Seq_Tham_vs_PHalo_raw[X1_X2_mask_STJ][Y1_Y2_mask_STJ]
RFC_Seq_lim_vs_GSE_raw_copy = RFC_Seq_lim_vs_GSE_raw[X1_X2_mask_SLG]
RFC_Seq_Tham_vs_GSE_raw_copy = RFC_Seq_Tham_vs_GSE_raw[X1_X2_mask_STG]


'IF I WANT TO REMOVE THE CASES WHERE PRECISION OR RECALL IS 1.0, THEN CALL THS IN,'
'IF I WANT CASES WHERE THEY ARE 1.0 I CAN CALL THEM OUT'
# Precision_recall_mask_GH = ((RFC_GSE_vs_PHalo_raw['prec GSE'] < 1.0) & (RFC_GSE_vs_PHalo_raw['recall P_Halo'] < 1.0))
# Precision_recall_mask_SLH = ((RFC_Seq_lim_vs_PHalo_raw['prec Seq_lim'] < 1.0) & (RFC_Seq_lim_vs_PHalo_raw['recall P_Halo'] < 1.0))
# Precision_recall_mask_STH = ((RFC_Seq_Tham_vs_PHalo_raw['prec Seq_Tham'] < 1.0) & (RFC_Seq_Tham_vs_PHalo_raw['recall P_Halo'] < 1.0))
# Precision_recall_mask_SLG = ((RFC_Seq_lim_vs_GSE_raw['prec Seq_lim'] < 1.0) & (RFC_Seq_lim_vs_GSE_raw['recall GSE'] < 1.0))
# Precision_recall_mask_STG = ((RFC_Seq_Tham_vs_GSE_raw['prec Seq_Tham'] < 1.0) & (RFC_Seq_Tham_vs_GSE_raw['recall GSE'] < 1.0))

# RFC_GSE_vs_PHalo_raw_copy = RFC_GSE_vs_PHalo_raw[X1_X2_mask_GH][Y1_Y2_mask_GH][Precision_recall_mask_GH]
# RFC_Seq_lim_vs_PHalo_raw_copy = RFC_Seq_lim_vs_PHalo_raw[X1_X2_mask_SLH][Y1_Y2_mask_SLH][Precision_recall_mask_SLH]
# RFC_Seq_Tham_vs_PHalo_raw_copy = RFC_Seq_Tham_vs_PHalo_raw[X1_X2_mask_STJ][Y1_Y2_mask_STJ][Precision_recall_mask_STH]
# RFC_Seq_lim_vs_GSE_raw_copy = RFC_Seq_lim_vs_GSE_raw[X1_X2_mask_SLG][Precision_recall_mask_SLG]
# RFC_Seq_Tham_vs_GSE_raw_copy = RFC_Seq_Tham_vs_GSE_raw[X1_X2_mask_STG][Precision_recall_mask_STG]

RFC_GSE_vs_PHalo_raw_copy

In [ ]:
RFC_test_dict = {
    # "Seq_lim vs Halo": RFC_Seq_lim_vs_PHalo_raw,
    # "Seq_Tham vs Halo": RFC_Seq_Tham_vs_PHalo_raw,
    # "1Seq_lim vs GSE": RFC_Seq_lim_vs_GSE_raw,
    # "1Seq_Tham vs GSE": RFC_Seq_Tham_vs_GSE_raw,
    # "GSE vs Halo_1": RFC_GSE_vs_PHalo_raw,

    'Seq_lim vs Halo': RFC_Seq_lim_vs_PHalo_raw_copy,
    "Seq_Tham vs Halo": RFC_Seq_Tham_vs_PHalo_raw_copy,
    "Seq_lim vs GSE": RFC_Seq_lim_vs_GSE_raw_copy,
    "Seq_Tham vs GSE": RFC_Seq_Tham_vs_GSE_raw_copy,
    "GSE vs Halo": RFC_GSE_vs_PHalo_raw_copy
}

print()
element_list = element_list_partial_small_galah_18
object_dict = {
    name: tests.copy()
    for name, tests in RFC_test_dict.items()
}
percents = [0.1]
for percent in percents:
    elm_frequency_table = {}
    for name, tests in object_dict.items():
        n = math.ceil(len(tests) * percent)
        print(f'{n} rows for {percent*100}% of the sample size ({len(tests)})')
        tests['element_list'] = (
                    tests['ratio1'].str.lower().str.split(r'[_/]+', regex=True)
                    + tests['ratio2'].str.lower().str.split(r'[_/]+', regex=True)
                    + tests['ratio3'].str.lower().str.split(r'[_/]+', regex=True)
                    )
        print(f"RFC test for {name}:")
        tests.sort_values(by='accuracy', ascending=False, inplace=True)
        print(n)
        percent_mask = tests['accuracy'] >= tests['accuracy'].iloc[n-1]
        tests = tests[percent_mask]
        n = len(tests)
        elm_frequency_table[name] = energy_tests_best_elements(tests, name, element_list, n=n, percents = True, bibtext_format = False)
        # display(HTML(elm_frequency_table[name].to_html(index=False)))
    elm_frequency_table = reduce(lambda left, right: pd.merge(left, right, on='elm', how='outer'), elm_frequency_table.values())

    costom_index = []
    # costom_index.append('xi_sq_statistic')
    costom_index.append('xi_sq_p_value')
    for elm in element_list:
        costom_index.append(elm)
    elm_frequency_table = (
        elm_frequency_table
        .set_index("elm")
        .reindex(costom_index)
        .reset_index()
    )

    # elm_frequency_table.sort_values(by='Seq_lim vs GSE frequency', ascending=False, inplace=True)
    # display(HTML(elm_frequency_table.to_html(index=False)))
    with pd.option_context('display.float_format', lambda x: f'{x:.1f}'):
        display(HTML(elm_frequency_table.to_html(index=False)))

In [ ]:
df1 = RFC_GSE_vs_PHalo_raw_copy
df2 = RFC_Seq_lim_vs_GSE_raw_copy
df3 = RFC_Seq_lim_vs_PHalo_raw_copy
df4 = RFC_Seq_Tham_vs_GSE_raw_copy
df5 = RFC_Seq_Tham_vs_PHalo_raw_copy
# df6 = RFC_Seq_Tham_vs_GSE_raw_copy_copy
col1 = 'accuracy'
# col1 = 'prec'
# col1 = 'recall'

col2 = 'prec'
# col2 = 'recall'

# plt.scatter((df1[col1]), df1[f'{col2} GSE'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'GSE vs Halo')
# plt.scatter((df2[col1]), df2[f'{col2} Seq_lim'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_lim vs GSE')
# plt.scatter((df3[col1]), df3[f'{col2} Seq_lim'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_lim vs Halo')
# plt.scatter((df4[col1]), df4[f'{col2} Seq_Tham'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_Tham vs GSE')
# plt.scatter((df5[col1]), df5[f'{col2} Seq_Tham'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_Tham vs Halo')


# plt.scatter((df1[f'{col1} P_Halo']), df1[f'{col2} GSE'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'GSE vs Halo')
# plt.scatter((df4[f'{col1} GSE']), df4[f'{col2} Seq_Tham'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_Tham vs GSE')
# plt.scatter((df5[f'{col1} P_Halo']), df5[f'{col2} Seq_Tham'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_Tham vs Halo')
# plt.scatter((df2[f'{col1} GSE']), df2[f'{col2} Seq_lim'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_lim vs GSE')
# plt.scatter((df3[f'{col1} P_Halo']), df3[f'{col2} Seq_lim'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_lim vs Halo')

# plt.scatter((df6[f'{col1} GSE']), df6[f'{col2} Seq_Tham'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_Tham vs GSE (copy)')

# plt.scatter((df1[f'{col1} GSE']), df1[f'{col2} GSE'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'GSE vs Halo')
# # plt.scatter((df4[f'{col1} Seq_Tham']), df4[f'{col2} Seq_Tham'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_Tham vs GSE')
# plt.scatter((df5[f'{col1} Seq_Tham']), df5[f'{col2} Seq_Tham'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_Tham vs Halo')
# # plt.scatter((df2[f'{col1} Seq_lim']), df2[f'{col2} Seq_lim'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_lim vs GSE')
# plt.scatter((df3[f'{col1} Seq_lim']), df3[f'{col2} Seq_lim'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_lim vs Halo')

plt.scatter((df1[col1]), df1[f'{col2} P_Halo'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'GSE vs Halo')
plt.scatter((df4[col1]), df4[f'{col2} GSE'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_Tham vs GSE')
plt.scatter((df5[col1]), df5[f'{col2} P_Halo'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_Tham vs Halo')
plt.scatter((df3[col1]), df3[f'{col2} P_Halo'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_lim vs Halo')
plt.scatter((df2[col1]), df2[f'{col2} GSE'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = 'Seq_lim vs GSE')


# df = df1
# X1_X2 = 'ti_fe'
# X1_X2_mask = ((df['ratio1'] == X1_X2) | (df['ratio2'] == X1_X2) | (df['ratio3'] == X1_X2))
# plt.scatter((df[X1_X2_mask][col1]), df[X1_X2_mask][f'{col2} GSE'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = f'{X1_X2} highlighted')

# df = df2
# X1 = 'nd_na'
# X1_X2_mask = (df['ratio1'].str.contains(X1, na=False) | df['ratio2'].str.contains(X1, na=False) | df['ratio3'].str.contains(X1, na=False))
# plt.scatter((df[X1_X2_mask][col1]), df[X1_X2_mask][f'{col2} Seq_lim'], s = 10, alpha = 0.8, cmap = 'coolwarm', label = f'{X1_X2} highlighted')

plt.xlabel(col1)
plt.ylabel(f'{col2} of the smaller group')
plt.legend( bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
# plt.title(f'RFC test: GSE vs Halo: {len(df)} combinations')#, {X1_X2} highlighted')

In [ ]:
# MINI CORNER PLOTS for individual use

# sample_set = RFC_Seq_lim_vs_PHalo_raw_copy
# sample_set = RFC_Seq_Tham_vs_PHalo_raw_copy 
# sample_set = RFC_Seq_lim_vs_GSE_raw_copy
# sample_set = RFC_Seq_Tham_vs_GSE_raw_copy
sample_set = RFC_GSE_vs_PHalo_raw_copy



names = [
    'Halo',
    'GSE',
    # 'Seq_Tham',
    # 'Seq_lim'
]

print("Number of samples:", len(sample_set))

bandwidth_percent = 0.075
N = 10
acc_l = "accuracy"
acc_r = False

# Select the top-ranked combinations
sample_set = sample_set.sort_values(by=acc_l, ascending=False)
TOP_N = top_min_corner_plots(sample_set, N)
TOP_labels = min_corner_plot_lables_comparison(
    TOP_N, N, acc_l, acc_r=acc_r
)

# Output directory
# save_folder = os.path.join(save_dir, folder_name)
# os.makedirs(save_folder, exist_ok=True)

total_plots = len(TOP_labels[0])

for i in range(total_plots):

    # Labels and accuracy
    labels = TOP_labels[0][i]
    e_labels = TOP_labels[1][i]
    accuracy_1 = TOP_labels[2][i]
    accuracy_2 = TOP_labels[3][i] if acc_r else None

    display_labels = [ratio_label(label) for label in labels]

    print(f"Creating plot {i + 1}/{total_plots}")
    print(f"plot {i + 1}/{total_plots}: {display_labels}")

    # Plot settings
    plot_datapoints = True
    plot_contours = False
    fill_contours = False
    plot_density = False

    # Input datasets:
    # (values, errors, color, marker size, alpha)
    corner_datasets = {
                "Halo": (galah_Gaia_Halo_new_ratios[labels],galah_Gaia_Halo_new_ratios[e_labels],Halo_C, 1, 0.6),
                "GSE": (galah_Gaia_GSE_new_ratios[labels],galah_Gaia_GSE_new_ratios[e_labels],GSE_C, 2.0, 0.6),
                "Seq_Tham": (galah_Gaia_Sequoia_p_Thamnos_new_ratios[labels],galah_Gaia_Sequoia_p_Thamnos_new_ratios[e_labels],Sequoia_C_2, 2.5, 0.6),
                "Seq_lim": (galah_Gaia_Sequoia_new_ratios[labels],galah_Gaia_Sequoia_new_ratios[e_labels],Sequoia_C, 2.5, 0.6),
                }

    corner_datasets = get_pair(corner_datasets, *names)

    # Convert dataframes to NumPy arrays and remove rows
    # where every value is non-finite.
    for key, (df_values, df_errors, color, ms, alpha) in (
        corner_datasets.items()
    ):
        values = df_values.to_numpy()
        errors = df_errors.to_numpy()

        row_mask = np.any(np.isfinite(values), axis=1)
        values = values[row_mask]
        errors = errors[row_mask]

        corner_datasets[key] = (
            values, errors, color, ms, alpha
        )

    # Use Halo as the initial dataset
    first_plot = next(iter(corner_datasets))
    first_values, first_errors, first_color, first_ms, first_alpha = (
        corner_datasets[first_plot]
    )

    ndim = first_values.shape[1]

    # Calculate common axis ranges using all datasets
    ranges = []

    for dim in range(ndim):
        combined = np.concatenate([
            values[:, dim]
            for values, errors, color, ms, alpha
            in corner_datasets.values()
        ])

        combined = combined[np.isfinite(combined)]

        if combined.size == 0:
            raise ValueError(
                f"No finite values found for dimension {dim}."
            )

        low, high = np.percentile(combined, [1, 99])
        padding = 0.05 * (high - low)

        # Avoid a zero-width range for constant-valued columns
        if padding == 0:
            padding = max(abs(low) * 0.05, 1e-6)

        ranges.append((low - padding, high + padding))

    # Create the initial corner plot
    fig = corner.corner(
        first_values,
        labels=display_labels,
        color=first_color,
        range=ranges,
        data_kwargs={
            "ms": first_ms,
            "alpha": first_alpha,
            "color": first_color,
        },
        hist_kwargs={
            "density": True,
            "histtype": "step",
            "linewidth": 0.5,
        },
        plot_datapoints=plot_datapoints,
        plot_contours=plot_contours,
        fill_contours=fill_contours,
        plot_density=plot_density,
    )

    # Add accuracy to the title
    ACC_1 = round(accuracy_1, 4)
    ACC_2 = round(accuracy_2, 4) if accuracy_2 is not None else None

    if ACC_2 is not None:
        fig.suptitle(
            f"Accuracy: {ACC_1} and {ACC_2}",
            fontsize=16,
            y=0.98,
        )
    else:
        fig.suptitle(
            f"Accuracy: {ACC_1}",
            fontsize=16,
            y=0.98,
        )

    axes = np.asarray(fig.axes).reshape((ndim, ndim))

    # Overlay the remaining datasets
    for key, (values, errors, color, ms, alpha) in (
        corner_datasets.items()
    ):
        if key == first_plot:
            continue

        corner.corner(
            values,
            fig=fig,
            color=color,
            range=ranges,
            labels=display_labels,
            data_kwargs={
                "ms": ms,
                "alpha": alpha,
                "color": color,
            },
            hist_kwargs={
                "density": True,
                "histtype": "step",
                "linewidth": 0.5,
            },
            plot_datapoints=plot_datapoints,
            plot_contours=plot_contours,
            fill_contours=fill_contours,
            plot_density=plot_density,
        )

    # Replace diagonal histograms with KDE curves.
    # Remove histogram patches only; do not clear the axes.
    for dim, ax in enumerate(axes.diagonal()):

        for patch in list(ax.patches):
            patch.remove()

        for values, errors, color, ms, alpha in (
            corner_datasets.values()
        ):
            x = values[:, dim]
            x = x[np.isfinite(x)]

            if x.size < 2:
                continue

            bandwidth = max(
                np.ptp(x) * bandwidth_percent,
                1e-6,
            )

            kde = KernelDensity(
                kernel="gaussian",
                bandwidth=bandwidth,
            )
            kde.fit(x.reshape(-1, 1))

            x_min, x_max = ranges[dim]
            x_grid = np.linspace(
                x_min, x_max, 1000
            ).reshape(-1, 1)

            density = np.exp(kde.score_samples(x_grid))
            peak = density.max()

            if peak > 0:
                density /= peak

            ax.plot(
                x_grid[:, 0],
                density,
                color=color,
                linewidth=1,
            )

        ax.set_xlim(ranges[dim])
        ax.set_ylim(0, 1.05)

    # Set all three bottom x-axis labels
    for col, label in enumerate(display_labels):
        axes[-1, col].set_xlabel(
            label,
            fontsize=12,
            labelpad=12,
        )

    # Set left-side y-axis labels
    for row, label in enumerate(display_labels):
        axes[row, 0].set_ylabel(
            label,
            fontsize=12,
            labelpad=10,
        )

    # # Save the figure
    # output_path = os.path.join(
    #     save_folder,
    #     f"plot_{i + 1}.png",
    # )

    # fig.savefig(
    #     output_path,
    #     dpi=300,
    #     bbox_inches="tight",
    #     pad_inches=0.15,
    # )

    # plt.close(fig)

    # print(f"Saved: {output_path}")

In [ ]:
# MINI CORNER PLOTS
datasets = {
    # "Seq_lim vs Halo": (RFC_Seq_lim_vs_PHalo_raw_copy, "RFC_Seq_lim_vs_PHalo_elmlist_18", 'Seq_lim', 'Halo', 'prec Seq_lim' , 'prec P_Halo', 'recall Seq_lim', 'recall P_Halo'),
    # "Seq_Tham vs Halo": (RFC_Seq_Tham_vs_PHalo_raw_copy, "RFC_Seq_Tham_vs_PHalo_elmlist_18", 'Seq_Tham', 'Halo', 'prec Seq_Tham' , 'prec P_Halo', 'recall Seq_Tham', 'recall P_Halo'),
    # "Seq_lim vs GSE": (RFC_Seq_lim_vs_GSE_raw_copy, "RFC_Seq_lim_vs_GSE_elmlist_18", 'Seq_lim', 'GSE', 'prec Seq_lim' , 'prec GSE', 'recall Seq_lim', 'recall GSE'),
    # "Seq_Tham vs GSE": (RFC_Seq_Tham_vs_GSE_raw_copy, "RFC_Seq_Tham_vs_GSE_elmlist_18", 'Seq_Tham', 'GSE', 'prec Seq_Tham' , 'prec GSE', 'recall Seq_Tham', 'recall GSE'),
    "GSE vs Halo": (RFC_GSE_vs_PHalo_raw_copy, "RFC_GSE_vs_PHalo_elmlist_18", 'GSE', 'Halo', 'prec GSE' , 'prec P_Halo', 'recall GSE', 'recall P_Halo')
}

name, (sample_set, folder_name, key1, key2, prec1, prec2, recall1, recall2) = next(iter(datasets.items()))
save_folder = os.path.join(save_dir, folder_name)
os.makedirs(save_folder, exist_ok=True)

names = [
    'Halo',
    'GSE',
    # 'Seq_Tham',
    # 'Seq_lim'
]

print("Number of samples:", len(sample_set))

bandwidth_percent = 0.075
N = 3
acc_l = "accuracy"
acc_r = False

# Select the top-ranked combinations
sample_set = (sample_set.sort_values(by=acc_l, ascending=False).reset_index(drop=True))
TOP_N = top_min_corner_plots(sample_set, N)
TOP_labels = min_corner_plot_lables_comparison(
    TOP_N, N, acc_l, acc_r=acc_r
)

# Output directory
save_folder = os.path.join(save_dir, folder_name)
os.makedirs(save_folder, exist_ok=True)

total_plots = len(TOP_labels[0])

for i in range(total_plots):

    # Labels and accuracy
    labels = TOP_labels[0][i]
    e_labels = TOP_labels[1][i]
    accuracy_1 = TOP_labels[2][i]
    accuracy_2 = TOP_labels[3][i] if acc_r else None
    precision_1 = sample_set.loc[i, prec1]
    precision_2 = sample_set.loc[i, prec2]
    recall_1 = sample_set.loc[i, recall1]
    recall_2 = sample_set.loc[i, recall2]

    display_labels = [ratio_label(label) for label in labels]

    print(f"Creating plot {i + 1}/{total_plots}")
    print(f"plot {i + 1}/{total_plots}: {display_labels}")

    # Plot settings
    plot_datapoints = True
    plot_contours = False
    fill_contours = False
    plot_density = False

    # Input datasets:
    # (values, errors, color, marker shape, marker size, alpha)
    corner_datasets = {
                "Halo": (galah_Gaia_Halo_new_ratios[labels],galah_Gaia_Halo_new_ratios[e_labels],Halo_C, Halo_shape, 1.5, 0.6),
                "GSE": (galah_Gaia_GSE_new_ratios[labels],galah_Gaia_GSE_new_ratios[e_labels],GSE_C, GSE_shape, 2, 0.6),
                "Seq_Tham": (galah_Gaia_Sequoia_p_Thamnos_new_ratios[labels],galah_Gaia_Sequoia_p_Thamnos_new_ratios[e_labels],Sequoia_C_2, Sequoia_shape_2, 4, 0.6),
                "Seq_lim": (galah_Gaia_Sequoia_new_ratios[labels],galah_Gaia_Sequoia_new_ratios[e_labels],Sequoia_C, Sequoia_shape, 3, 0.6),
                }

    corner_datasets = get_pair(corner_datasets, *names)

    # Convert dataframes to NumPy arrays and remove rows
    # where every value is non-finite.
    for key, (df_values, df_errors, color, shape, ms, alpha) in (
        corner_datasets.items()
    ):
        values = df_values.to_numpy()
        errors = df_errors.to_numpy()

        row_mask = np.any(np.isfinite(values), axis=1)
        values = values[row_mask]
        errors = errors[row_mask]

        corner_datasets[key] = (
            values, errors, color, shape, ms, alpha
        )

    # Use Halo as the initial dataset
    first_plot = next(iter(corner_datasets))
    first_values, first_errors, first_color, first_shape, first_ms, first_alpha = (
        corner_datasets[first_plot]
    )

    ndim = first_values.shape[1]

    # Calculate common axis ranges using all datasets
    ranges = []

    for dim in range(ndim):
        combined = np.concatenate([
            values[:, dim]
            for values, errors, color, shape, ms, alpha
            in corner_datasets.values()
        ])

        combined = combined[np.isfinite(combined)]

        if combined.size == 0:
            raise ValueError(
                f"No finite values found for dimension {dim}."
            )

        low, high = np.percentile(combined, [1, 99])
        padding = 0.05 * (high - low)

        # Avoid a zero-width range for constant-valued columns
        if padding == 0:
            padding = max(abs(low) * 0.05, 1e-6)

        ranges.append((low - padding, high + padding))

    # Create the initial corner plot
    fig = corner.corner(
        first_values,
        labels=display_labels,
        color=first_color,
        range=ranges,
        data_kwargs={
            "marker": first_shape,
            "ms": first_ms,
            "alpha": first_alpha,
            "color": first_color,
        },
        hist_kwargs={
            "density": True,
            "histtype": "step",
            "linewidth": 0.5,
        },
        plot_datapoints=plot_datapoints,
        plot_contours=plot_contours,
        fill_contours=fill_contours,
        plot_density=plot_density,
    )

    # Add accuracy to the title
    ACC_1 = f'{accuracy_1:.3f}'
    ACC_2 = f'{accuracy_2:.3f}' if accuracy_2 is not None else None
    PREC_1 = f'{precision_1:.3f}'
    PREC_2 = f'{precision_2:.3f}'
    REC_1 = f'{recall_1:.3f}'
    REC_2 = f'{recall_2:.3f}'

    if ACC_2 is not None:
        fig.suptitle(
            f"{name}: Accuracy: {ACC_1} and {ACC_2}",
            fontsize=16,
            y=0.98,
        )
    else:
        fig.suptitle(
            f"{name}:Accuracy: {ACC_1}",
            fontsize=16,
            y=0.98,
        )

    axes = np.asarray(fig.axes).reshape((ndim, ndim))

    # Overlay the remaining datasets
    handles = []
    for key, (values, errors, color, shape, ms, alpha) in (
        corner_datasets.items()
    ):
        print(key)
        handles.append(
            plt.Line2D(
                [0],
                [0],
                color=color,
                label=f"{key}: Precision: {PREC_1}, Recall: {REC_1}" if key == key1 else f"{key}: Precision: {PREC_2}, Recall: {REC_2}" if key == key2 else f"{key}",
                marker=shape,
                linewidth=2,
            )
        )
        if key == first_plot:
            continue

        corner.corner(
            values,
            fig=fig,
            color=color,
            range=ranges,
            labels=display_labels,
            data_kwargs={
                "marker": shape,
                "ms": ms,
                "alpha": alpha,
                "color": color,
            },
            hist_kwargs={
                "density": True,
                "histtype": "step",
                "linewidth": 0.5,
            },
            plot_datapoints=plot_datapoints,
            plot_contours=plot_contours,
            fill_contours=fill_contours,
            plot_density=plot_density,
        )

    # Replace diagonal histograms with KDE curves.
    # Remove histogram patches only; do not clear the axes.
    for dim, ax in enumerate(axes.diagonal()):

        for patch in list(ax.patches):
            patch.remove()

        for values, errors, color, shape, ms, alpha in (
            corner_datasets.values()
        ):
            x = values[:, dim]
            x = x[np.isfinite(x)]

            if x.size < 2:
                continue

            bandwidth = max(
                np.ptp(x) * bandwidth_percent,
                1e-6,
            )

            kde = KernelDensity(
                kernel="gaussian",
                bandwidth=bandwidth,
            )
            kde.fit(x.reshape(-1, 1))

            x_min, x_max = ranges[dim]
            x_grid = np.linspace(
                x_min, x_max, 1000
            ).reshape(-1, 1)

            density = np.exp(kde.score_samples(x_grid))
            peak = density.max()

            if peak > 0:
                density /= peak

            ax.plot(
                x_grid[:, 0],
                density,
                color=color,
                linewidth=1,
            )

        ax.set_xlim(ranges[dim])
        ax.set_ylim(0, 1.05)

    # Set all three bottom x-axis labels
    for col, label in enumerate(display_labels):
        axes[-1, col].set_xlabel(
            label,
            fontsize=12,
            labelpad=12,
        )

    # Set left-side y-axis labels
    for row, label in enumerate(display_labels):
        axes[row, 0].set_ylabel(
            label,
            fontsize=12,
            labelpad=10,
        )

    fig.legend(
    handles=handles,
    loc="upper left",
    bbox_to_anchor=(0.5, 0.9), # x-coordinate, y-coordinate
    fontsize=12,
    )
    # Save the figure
    output_path = os.path.join(
        save_folder,
        f"plot_{i + 1}.pdf",
    )

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.15,
    )

    # plt.show(fig)
    plt.close(fig)

    # print(f"Saved: {output_path}")